# ACIT4620 — Notebook 06: Shared EfficientNet-B0 Feature Extraction

**Purpose:** Load the *verified E2a-50 checkpoint* and extract **identical frozen 1,280-D pooled EfficientNet-B0 features** for train and validation. These features will later be used by a newly trained linear head, E3 MLP and E4 first-order Takagi–Sugeno ANFIS (R = 4, 8, 16).

**Research safeguards:** Original 10,015-image HAM10000 EDA manifest, seven-class ordering, strict lesion-disjoint split and SHA-256 verification, deterministic evaluation preprocessing, no new split, **no training**, **no test-set feature extraction or evaluation**, and no use of calibrated class probabilities as neuro-fuzzy inputs.

**Kaggle inputs:** (1) `kmader/skin-cancer-mnist-ham10000`, (2) `athenaanalyzes/ham10000-eda-7class` saved outputs, (3) private `ACIT4620-E2a50-Model-Artifacts` dataset containing `best_efficientnet_b0.pth`.

**Note:** The shared CNN backbone was already selected using validation data in the earlier E2a-50 experiment. All head comparisons will use this frozen backbone; the saved E1 end-to-end baseline is a separate reference experiment. Only selected, fixed-head models will be assessed on the held-out test set in a later final stage.

## 1. Environment, imports, and immutable configuration

In [ ]:
from pathlib import Path
import hashlib, json, time, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0
from PIL import Image, ImageOps

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
CLASS_CODES = ['nv','mel','bkl','bcc','akiec','vasc','df']
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_CODES)}
EXPECTED_SPLIT_SHA256 = '265ec2412f4dd7d50c75b46d75a095c0d3445250895b90b91f3d73aca9aebce0'
EXPECTED_SPLIT_SIZES = {'train': 7116, 'validation': 1441, 'test': 1458}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64   # Safe to lower to 32 or 16 if needed; does not alter extracted features.
NUM_WORKERS = 2
OUT = Path('/kaggle/working/ACIT4620_shared_features_e2a50')
OUT.mkdir(parents=True, exist_ok=True)
INPUT = Path('/kaggle/input')
print('PyTorch:',torch.__version__,'Device:',DEVICE)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print('Output folder:',OUT)


## 2. Locate exactly one EDA manifest, audit, raw dataset, and E2a-50 checkpoint

In [ ]:
def exactly_one(filename, required_fragment=None):
    found = sorted(INPUT.rglob(filename))
    if required_fragment:
        found = [p for p in found if required_fragment.lower() in str(p).lower()]
    if len(found) != 1:
        raise RuntimeError(f'Expected exactly one {filename}, got {len(found)}: {found}. Check attached inputs.')
    return found[0]

MANIFEST_PATH = exactly_one('seven_class_master_manifest.csv')
AUDIT_PATH = exactly_one('audit_summary.json')
METADATA_PATH = exactly_one('HAM10000_metadata.csv')
CHECKPOINT_PATH = exactly_one('best_efficientnet_b0.pth')
DATA_DIR = METADATA_PATH.parent
for label, path in [('Manifest',MANIFEST_PATH),('Audit',AUDIT_PATH),('Images root',DATA_DIR),('E2a-50 checkpoint',CHECKPOINT_PATH)]:
    print(label,':',path)
assert 'e2a50' in str(CHECKPOINT_PATH).lower() or 'e2a_50' in str(CHECKPOINT_PATH).lower(), 'Checkpoint is not clearly from E2a-50 private artifacts.'


## 3. Verify full EDA audit, class encoding and lesion-disjoint split

In [ ]:
audit = json.loads(AUDIT_PATH.read_text())
assert audit.get('full_image_audit') is True
assert audit.get('exact_hash_audit_completed') is True
assert audit.get('images_quality_audited') == 10015
assert audit.get('quality_audit_failures') == 0
assert audit.get('split_lesion_overlap') == 0

manifest = pd.read_csv(MANIFEST_PATH)
assert len(manifest) == 10015 and manifest['image_id'].is_unique
assert manifest['lesion_id'].notna().all()
assert set(manifest['dx']) == set(CLASS_CODES)
assert set(manifest['split']) == set(EXPECTED_SPLIT_SIZES)
assert manifest.groupby('lesion_id')['split'].nunique().max() == 1, 'Lesion ID leakage!'
assert manifest['dx'].map(CLASS_TO_IDX).notna().all()
if 'label' in manifest.columns:
    assert (manifest['label'].astype(int).to_numpy() == manifest['dx'].map(CLASS_TO_IDX).astype(int).to_numpy()).all(), 'Label ordering mismatch'
manifest['label'] = manifest['dx'].map(CLASS_TO_IDX).astype(int)
for name, expected in EXPECTED_SPLIT_SIZES.items():
    assert int((manifest.split == name).sum()) == expected

assignment_string = '\n'.join(manifest[['image_id','lesion_id','dx','split']].astype(str).sort_values('image_id').agg('|'.join,axis=1))
split_hash = hashlib.sha256(assignment_string.encode()).hexdigest()
assert split_hash == EXPECTED_SPLIT_SHA256, f'Split changed: {split_hash}'
print('PASS — full audit, class order, lesion-disjoint split, assignment SHA-256')
print('Split counts:',manifest['split'].value_counts().to_dict())
print('Split SHA-256:',split_hash)


## 4. Resolve original image files — do not create new splits

In [ ]:
paths = {}
for folder in (DATA_DIR/'HAM10000_images_part_1', DATA_DIR/'HAM10000_images_part_2'):
    assert folder.is_dir(), f'Missing image folder {folder}'
    for p in folder.glob('*.jpg'):
        assert p.stem not in paths, f'Duplicate raw image identifier: {p.stem}'
        paths[p.stem] = p
assert len(paths) == 10015
assert all(image_id in paths for image_id in manifest.image_id)
manifest['resolved_path'] = [str(paths[image_id]) for image_id in manifest.image_id]
parts = {k: manifest.loc[manifest['split'].eq(k)].copy().reset_index(drop=True) for k in ['train','validation']}
print('Resolved',len(paths),'raw image paths')
print('Train:',len(parts['train']),'Validation:',len(parts['validation']))
print('Test images NOT passed to any dataset or DataLoader.')


## 5. Load and verify the validation-selected EfficientNet-B0 checkpoint (no downloads, no retraining)

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert ckpt['class_codes'] == CLASS_CODES, f'Checkpoint class ordering differs: {ckpt["class_codes"]}'
assert ckpt['split_sha256'] == split_hash, 'Checkpoint uses different split'

cnn = efficientnet_b0(weights=None)  # Do not download weights: checkpoint contains trained weights.
in_features = cnn.classifier[1].in_features
assert in_features == 1280
cnn.classifier[1] = nn.Linear(in_features, len(CLASS_CODES))
cnn.load_state_dict(ckpt['model_state'], strict=True)
cnn = cnn.to(DEVICE).eval()
for p in cnn.parameters(): p.requires_grad_(False)
assert not any(p.requires_grad for p in cnn.parameters())
print('PASS — checkpoint loaded strictly, class order and split verified')
print('Checkpoint validation macro-F1:',ckpt.get('val_macro_f1'))
print('Checkpoint validation balanced accuracy:',ckpt.get('val_balanced_accuracy'))
print('Shared representation: cnn.features -> cnn.avgpool -> flattened 1280-D features')


## 6. Deterministic ImageNet preprocessing for **both** feature partitions

In [ ]:
MEAN = [0.485,0.456,0.406]
STD = [0.229,0.224,0.225]
feature_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD)
])

class FeatureImageDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row['resolved_path']) as image:
            x = feature_transform(ImageOps.exif_transpose(image).convert('RGB'))
        return x, int(row['label']), row['image_id']

loaders = {
    key: DataLoader(FeatureImageDataset(frame), batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
    for key,frame in parts.items()
}
print('Frozen eval transforms:',feature_transform)
print('No random augmentation, no shuffled feature rows.')


## 7. Extract pooled 1,280-D features from the same CNN for train and validation

In [ ]:
@torch.inference_mode()
def extract_features(loader):
    feats, labels, image_ids = [],[],[]
    cnn.eval()
    start=time.perf_counter()
    for images, y, ids in loader:
        images = images.to(DEVICE, non_blocking=True)
        fmap = cnn.features(images)
        pooled = cnn.avgpool(fmap).flatten(1)
        assert pooled.shape[1] == 1280, pooled.shape
        feats.append(pooled.cpu().numpy().astype(np.float32))
        labels.append(y.numpy().astype(np.int64))
        image_ids.extend(list(ids))
    X=np.concatenate(feats,axis=0)
    y=np.concatenate(labels,axis=0)
    assert X.shape == (len(loader.dataset),1280)
    assert np.isfinite(X).all()
    return X,y,np.asarray(image_ids,dtype=str),time.perf_counter()-start

extracted = {}
for split in ['train','validation']:
    X,y,ids,seconds=extract_features(loaders[split])
    reference=parts[split].set_index('image_id').loc[ids]
    assert len(set(ids)) == len(ids)
    assert (reference['label'].to_numpy() == y).all(), f'{split}: labels do not match IDs'
    extracted[split] = (X,y,ids,reference,seconds)
    print(f'{split}: features={X.shape}, labels={y.shape}, unique lesions={reference.lesion_id.nunique()}, extract_seconds={seconds:.2f}')
assert set(extracted['train'][2]).isdisjoint(set(extracted['validation'][2]))
assert set(extracted['train'][3].lesion_id).isdisjoint(set(extracted['validation'][3].lesion_id))
print('PASS — no train/validation image-ID or lesion-ID overlap')


## 8. Save shareable arrays and aligned metadata for all head comparisons

In [ ]:
saved={}
for split,(X,y,ids,frame,seconds) in extracted.items():
    # String dtype (not object) permits safe NumPy loading with allow_pickle=False.
    lesion_ids = frame['lesion_id'].astype(str).to_numpy(dtype=str)
    class_names = frame['dx'].astype(str).to_numpy(dtype=str)
    path = OUT / f'{split}_features.npz'
    np.savez_compressed(path,
        X=X, y=y, image_id=ids, lesion_id=lesion_ids, dx=class_names)
    metadata = frame.reset_index()[['image_id','lesion_id','dx','label']].copy()
    metadata.to_csv(OUT / f'{split}_feature_manifest.csv',index=False)
    saved[split]=str(path)
    # Round-trip test, no pickle required.
    with np.load(path, allow_pickle=False) as loaded:
        assert np.array_equal(loaded['y'], y)
        assert np.array_equal(loaded['image_id'], ids)
        assert loaded['X'].shape == (len(y),1280)
    print('Saved:',path,'MB:',round(path.stat().st_size/1048576,2))

checkpoint_hash = hashlib.sha256()
with CHECKPOINT_PATH.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024*1024), b''):
        checkpoint_hash.update(chunk)
settings = {
    'experiment':'ACIT4620_06_Shared_EfficientNet_Feature_Extraction',
    'source_model':'E2a-50 validation-selected EfficientNet-B0',
    'source_checkpoint_file':str(CHECKPOINT_PATH),
    'checkpoint_sha256':checkpoint_hash.hexdigest(),
    'split_sha256':split_hash,
    'class_codes':CLASS_CODES,
    'feature_dimension':1280,
    'preprocessing':'EXIF transpose, RGB, Resize(256), CenterCrop(224), ToTensor, ImageNet normalization',
    'augmentation_for_extraction':'none; deterministic on train and validation',
    'backbone_frozen':True,
    'test_set_extracted':False,
    'head_training_performed':False,
    'feature_files':saved,
    'extract_seconds':{k:float(v[4]) for k,v in extracted.items()},
    'note':'One fixed CNN representation for linear, MLP and multi-output first-order Takagi-Sugeno ANFIS; project test remains locked.'
}
(OUT/'feature_extraction_config.json').write_text(json.dumps(settings,indent=2))
print('PASS — feature arrays and provenance saved')


## 9. Final output audit and download bundle

In [ ]:
import shutil
expected = ['train_features.npz','validation_features.npz','train_feature_manifest.csv',
            'validation_feature_manifest.csv','feature_extraction_config.json']
for filename in expected:
    p=OUT/filename
    assert p.is_file() and p.stat().st_size>0, f'Missing/empty {filename}'
    print('PASS',filename,round(p.stat().st_size/1048576,2),'MB')
archive=shutil.make_archive('/kaggle/working/ACIT4620_06_Shared_Features','zip',root_dir=OUT.parent,base_dir=OUT.name)
print('Download ZIP:',archive)
print('Zip size MB:',round(Path(archive).stat().st_size/1048576,2))
print('Ready to train linear, MLP, and ANFIS heads — NOT evaluated on held-out test.')


## Next in the project

1. Save this notebook with **Save Version → Save & Run All** (no training is included), and download the ZIP from Kaggle Output.
2. In the next Kaggle notebook, attach this notebook's saved outputs, use `np.load(..., allow_pickle=False)` and **fit feature scaling on training features only**, applying the same parameters to validation.
3. Train linear and **parameter-comparable** MLP baselines and first-order Takagi–Sugeno ANFIS heads with trainable `1280 → d` projection (initially `d=8`; rule counts 4, 8, 16). Match tuning budgets, use validation macro-F1 for selection, and report per-class metrics.
4. The shared features are cached/frozen. For later Grad-CAM, reload the original E2a-50 checkpoint and the final head weights in an end-to-end differentiable wrapper; cached features alone cannot produce image-level Grad-CAM gradients.
5. Do not use the held-out test images, labels, or features until **all** model selection is completed.